# Neural Candidate Generation - Two Tower Model

Implemented a content-aware two-tower neural network model for candidate generation. This model learns user preference from their click history and identifies relevant articles using their text and entity embeddings, together with historical engagement features, such as hourly article exposure and click counts, to capture changes in article popularity over time. These features are calculated using only past interactions to prevent data leakage. 

The model is trained on the full training dataset using predefined hyperparameters and evaluated by retrieving the Top-M articles from all articles available at each recommendation timestamp

## Step 1: Setup and Reproducibility

Import the required libraries, define the default model parameters, and set random seeds to ensure reproducibility. Configure the input directories for Stages 1, 3, and 4, and create the necessary output folders to store model checkpoints, training results, and evaluation outputs.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import random
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#Model hyperparameters
@dataclass(frozen=True)
class Config:
    EMBED_DIM: int = 128    #embedding dimension
    MAX_HISTORY: int = 50
    MAX_RECENT: int = 10
    NEGATIVES_PER_POS: int = 4 #negative sampling ratio
    BATCH_SIZE: int = 2048            
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-5
    FINAL_MAX_EPOCHS: int = 20
    EARLY_STOP_PATIENCE: int = 3
    USER_DROPOUT: float = 0.20 #randomly disables 20% of hidden activations in the user tower
    ITEM_DROPOUT: float = 0.10 #randomly disables 10% of hidden activations in the item tower
    USE_SELF_ATTENTION: bool = True
    USE_RECENT: bool = True
    USE_ATTENTION_POOLING: bool = True
    USE_HOURLY: bool = True
    USE_INBATCH_NEGATIVES: bool = True    
    UNIFORM_NEGATIVES: int = 512          
    LEARN_TEMPERATURE: bool = True
    INIT_TEMPERATURE: float = 0.05
    TOP_MS: tuple[int, ...] = (50, 100, 200, 500)
    SELECTION_M: int = 500
    EXCLUDE_HISTORY: bool = True #exclude recommending articles that the user has already interacted with in the past

CFG = Config()

def find_repo_root(start=Path.cwd()):
    for folder in (start, *start.parents):
        if ((folder / '3_Representations').is_dir() and
                (folder / '4_Feature_Engineering').is_dir() and
                (folder / '1_Data_Preparation').is_dir()):
            return folder
    raise FileNotFoundError('Incorrect Directory')

REPO_ROOT = find_repo_root()
STAGE1 = REPO_ROOT / '1_Data_Preparation' / 'processed'
STAGE3 = REPO_ROOT / '3_Representations' / 'artifacts'
STAGE4 = REPO_ROOT / '4_Feature_Engineering' / 'artifacts' / 'prequential-all'

NOTEBOOK_NAME = '5_Neural_Candidate_Generation_TT.ipynb'
NOTEBOOK_CANDIDATES = [
    REPO_ROOT / '5_Candidate_Generation' / 'Neural_Models',
    REPO_ROOT / '5_Models' / '5_01_Candidate_Generation' / 'nn_model',
    Path.cwd().resolve(),
]
notebook_matches = list(dict.fromkeys(
    p for p in NOTEBOOK_CANDIDATES if (p / NOTEBOOK_NAME).is_file()))
if len(notebook_matches) == 1:
    NOTEBOOK_DIR = notebook_matches[0]
elif Path.cwd().name == 'Neural_Models':
    NOTEBOOK_DIR = Path.cwd().resolve()
else:
    raise FileNotFoundError('Run this notebook from the folder containing its .ipynb file.')

MODEL_DIR = NOTEBOOK_DIR / 'artifacts'
DEFAULT_DIR = MODEL_DIR / 'default_hourly'
SHARED_DIR = MODEL_DIR / 'shared'
for folder in (DEFAULT_DIR, SHARED_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CTR_STRENGTH = 10.0
USE_LOGQ = True                          
HISTORY_ITEM_DROPOUT = 0.0
HISTORY_DROPOUT = 0.0
GRAD_CLIP = 1.0
SERVE_BATCH = 256
TEMPERATURE_RANGE = (0.02, 0.5)
SPLITS_USED = ('train', 'validation')
HOUR_NS = 3_600 * 10**9

required_logs = [STAGE1 / f'{kind}_{split}.parquet'
                 for kind in ('impressions', 'candidates') for split in SPLITS_USED]
missing_logs = [p for p in required_logs if not p.is_file()]
if missing_logs:
    raise FileNotFoundError(f'Missing Stage 1 logs needed for hourly features: {missing_logs}')
first_timestamp = pl.scan_parquet(STAGE1 / 'candidates_train.parquet').select(
    pl.col('timestamp').min()).collect().item()
last_timestamp = pl.scan_parquet(STAGE1 / 'candidates_validation.parquet').select(
    pl.col('timestamp').max()).collect().item()
T0_NS = int(pd.Timestamp(first_timestamp).floor('D').value)
N_HOURS = int((pd.Timestamp(last_timestamp).value - T0_NS) // HOUR_NS) + 1
if N_HOURS <= 0:
    raise ValueError('Invalid train/validation timestamp range')

print('Torch:', torch.__version__, '| Device:', DEVICE)
print('Stage 1:', STAGE1)
print('Stage 3:', STAGE3)
print('Stage 4:', STAGE4)
print('Outputs:', DEFAULT_DIR)
print('Hourly states:', N_HOURS, '| Hourly feature: ENABLED')


Torch: 2.14.1+cpu | Device: cpu
Stage 1: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\1_Data_Preparation\processed
Stage 3: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\3_Representations\artifacts
Stage 4: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\4_Feature_Engineering\artifacts\prequential-all
Outputs: C:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\nn_model\artifacts\default_hourly
Hourly states: 144 | Hourly feature: ENABLED


## 2 · Data checks and storage helpers

Define data-quality checks, variable-length history utilities, and a batched Parquet writer for candidate recommendations.


In [ ]:
QC = []

#validate data quality check
def check(area, name, value, expected=None, status=None, detail=""):
    if status is None:
        status = "PASS" if (expected is None or value == expected) else "FAIL"
    QC.append({"area": area, "check": name, "value": str(value),
               "expected": str(expected) if expected is not None else "",
               "status": status, "detail": detail})
    if status == "FAIL":
        raise ValueError(f"{area}: {name} | observed={value} expected={expected} {detail}")
    return value

def report_qc():
    return pd.DataFrame(QC)

# Retrieve all Parquet files from a directory
def parquet_files(path):
    path = Path(path)
    files = sorted(path.rglob("*.parquet")) if path.is_dir() else [path]
    if not files:
        raise FileNotFoundError(f"No Parquet files under {path}")
    return files


def list_to_csr(series):
    lengths = series.list.len().fill_null(0).to_numpy().astype(np.int64)
    flat = series.rechunk().to_arrow().flatten().to_numpy(zero_copy_only=False)
    if flat.size != lengths.sum():
        raise ValueError("list column contains null elements")
    offsets = np.zeros(len(lengths) + 1, dtype=np.int64)
    np.cumsum(lengths, out=offsets[1:])
    return offsets, flat

def csr_gather(offsets, flat, idx):
    idx = np.asarray(idx, dtype=np.int64)
    starts = offsets[idx]
    lengths = offsets[idx + 1] - starts
    total = int(lengths.sum())
    owner = np.repeat(np.arange(len(idx), dtype=np.int64), lengths)
    within = np.arange(total, dtype=np.int64) - np.repeat(np.cumsum(lengths) - lengths, lengths)
    return flat[np.repeat(starts, lengths) + within].astype(np.int64), owner, lengths

#right padding to ensure fixed length sequences
def padded_tail(series, width):
    offsets, flat = list_to_csr(series.list.tail(width))
    lengths = np.diff(offsets)
    out = np.zeros((len(lengths), width), dtype=np.int32)
    out[np.repeat(np.arange(len(lengths)), lengths),
        np.arange(len(flat)) - np.repeat(offsets[:-1], lengths)] = flat
    return out


CANDIDATE_SCHEMA = pa.schema([("impression_uid", pa.int64()),
                              ("cutoff", pa.int16()),
                              ("news_code", pa.int32())])


class CandidateWriter:
    """Writes impression_uid, cutoff, news_code in batches (same as the baselines)."""

    def __init__(self, path, batch_rows=2_000_000):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.batch_rows, self.frames, self.pending, self.rows = batch_rows, [], 0, 0
        self.writer = pq.ParquetWriter(self.path, CANDIDATE_SCHEMA, compression="zstd")

    def add(self, impression_uid, cutoff, news_code):
        frame = pa.table({"impression_uid": pa.array(impression_uid, type=pa.int64()),
                          "cutoff": pa.array(cutoff, type=pa.int16()),
                          "news_code": pa.array(news_code, type=pa.int32())},
                         schema=CANDIDATE_SCHEMA)
        self.frames.append(frame)
        self.pending += frame.num_rows
        self.rows += frame.num_rows
        if self.pending >= self.batch_rows:
            self.flush()

    def flush(self):
        if self.frames:
            self.writer.write_table(pa.concat_tables(self.frames))
            self.frames, self.pending = [], 0

    def close(self):
        self.flush()
        self.writer.close()

print("Storage helpers ready")


Storage helpers ready


## Step 3: Load and Validate Article Features

Locate the required Stage 3 and Stage 4 input files, including article representations, availability timestamps, user profiles, training labels, and negative samples.

Load the precomputed text and entity embeddings, along with article category and subcategory information. Perform data-quality checks to ensure that article records, embeddings, and identifiers are correctly aligned.

Additionally, prepare article ID mappings and availability timestamps to ensure that only articles available at each recommendation time are eligible for retrieval. Finally, convert the article features into PyTorch tensors for neural network processing.

In [ ]:
REAL_ARTICLES = STAGE3 / "article_index.parquet"
REAL_AVAILABILITY = STAGE3 / "article_availability.parquet"

#locate stage 4 directory
def stage4_split_path(prefix, split):
    suffixes = ("validation", "val") if split == "validation" else (split,)
    matches = [STAGE4 / f"{prefix}_{suffix}" for suffix in suffixes
               if (STAGE4 / f"{prefix}_{suffix}").exists()]
    if len(matches) > 1:
        raise ValueError(f"Multiple {split} directories for {prefix}: {matches}")
    return matches[0] if matches else STAGE4 / f"{prefix}_{suffixes[0]}"

# Define Stage 4 input paths for user profiles, labels, and training negatives
REAL_PROFILES = {s: stage4_split_path("request_profiles", s)
                 for s in ("train", "validation", "test")}
REAL_LABELS = {s: stage4_split_path("labels", s)
               for s in ("train", "validation", "test")}
REAL_NEGATIVE_GROUPS = STAGE4 / "training_negative_groups"

#check whether the required Stage 3 and Stage 4 input files exist
print("Group 6 inputs (based on your confirmed paths):")
for p in [STAGE3 / "text64.npy", STAGE3 / "article_entity100.npy",
          REAL_ARTICLES, REAL_AVAILABILITY,
          *(REAL_PROFILES[s] for s in SPLITS_USED), *(REAL_LABELS[s] for s in SPLITS_USED), REAL_NEGATIVE_GROUPS]:
    print("  ", "OK" if p.exists() else "MISSING", p)

#display directory
for directory in (STAGE3, STAGE4):
    if directory.exists():
        print("\nAvailable paths under", directory)
        for p in sorted(directory.iterdir())[:20]:
            print("  ", p.name)

if STAGE4.exists():
    import pyarrow.dataset as ds
    for path in [REAL_PROFILES["train"], REAL_LABELS["train"],
                 REAL_NEGATIVE_GROUPS, STAGE4 / "ranking_features_train"]:
        if path.exists():
            print("\nSCHEMA", path.name, "→", ds.dataset(str(path), format="parquet").schema.names)

#load article metadata and retain the identifiers and categorical features required by the model
article = pl.read_parquet(REAL_ARTICLES)
for column in ("article_row", "news_code", "category_code", "subcategory_code"):
    check("articles", f"article_index has {column}", column in article.columns, True)
article = article.select("article_row", "news_code", "category_code", "subcategory_code").sort("article_row")

text = np.array(np.load(STAGE3 / "text64.npy"), dtype=np.float32)
entity = np.array(np.load(STAGE3 / "article_entity100.npy"), dtype=np.float32)
N_ROWS = len(text)                     

check("articles", "text dimension", text.shape[1], 64)
check("articles", "entity dimension", entity.shape[1], 100)
check("articles", "text/entity row alignment", len(entity), N_ROWS)
check("articles", "embedding rows = article records + padding", N_ROWS, article.height + 1)
check("articles", "article rows exactly 1..N",
      bool(np.array_equal(article["article_row"].to_numpy(), np.arange(1, N_ROWS))), True)
check("articles", "row 0 is an all-zero padding vector", bool(not text[0].any() and not entity[0].any()), True)
check("articles", "unique news codes", article["news_code"].n_unique(), article.height)

#merge article availability timestamps and discovery sources with the article metadata
availability = pl.read_parquet(REAL_AVAILABILITY, columns=["news_code", "available_from", "discovery_source"])
article = article.join(availability, on="news_code", how="left")
check("articles", "articles without available_from", int(article["available_from"].null_count()), 0)

rows_np = article["article_row"].to_numpy().astype(np.int64)
codes_np = article["news_code"].to_numpy().astype(np.int64)

cat_code = np.zeros(N_ROWS, dtype=np.int64)
sub_code = np.zeros(N_ROWS, dtype=np.int64)
cat_code[rows_np] = article["category_code"].fill_null(0).to_numpy()
sub_code[rows_np] = article["subcategory_code"].fill_null(0).to_numpy()
N_CAT, N_SUB = int(cat_code.max()) + 1, int(sub_code.max()) + 1

avail_ns = np.full(N_ROWS, np.iinfo(np.int64).max, dtype=np.int64)  
avail_ns[rows_np] = article.select(pl.col("available_from").dt.epoch("ns"))["available_from"].to_numpy()
history_only = np.zeros(N_ROWS, dtype=np.float32)
history_only[rows_np] = (article["discovery_source"] == "history").fill_null(False).to_numpy()


row_of_code = np.zeros(int(codes_np.max()) + 1, dtype=np.int64)
row_of_code[codes_np] = rows_np
row_to_code = np.zeros(N_ROWS, dtype=np.int64)
row_to_code[rows_np] = codes_np

#sort articles by availability timestamp
rows_by_avail = rows_np[np.lexsort((codes_np, avail_ns[rows_np]))]
avail_sorted = avail_ns[rows_by_avail]


def eligible_count(hour):
    return int(np.searchsorted(avail_sorted, T0_NS + (hour + 1) * HOUR_NS, side="left"))



TEXT_T = torch.from_numpy(text).to(DEVICE)
ENTITY_T = torch.from_numpy(entity).to(DEVICE)
CAT_T = torch.from_numpy(cat_code).to(DEVICE)
SUB_T = torch.from_numpy(sub_code).to(DEVICE)
AVAIL_T = torch.from_numpy(avail_ns).to(DEVICE)

print(f"Articles: {N_ROWS - 1:,} | categories incl. unseen: {N_CAT} | subcategories incl. unseen: {N_SUB}")
print(f"Articles with an entity vector: {int((np.abs(entity).sum(1) > 0).sum()):,}")
print(f"History-only discoveries: {int(history_only.sum()):,}")


Group 6 inputs (based on your confirmed paths):
   OK c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\3_Representations\artifacts\text64.npy
   OK c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\3_Representations\artifacts\article_entity100.npy
   OK c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\3_Representations\artifacts\article_index.parquet
   OK c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\3_Representations\artifacts\article_availability.parquet
   OK c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\4_Feature_Engineering\artifacts\prequential-all\request_profiles_train
   OK c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\4_Feature_Engineering\artifacts\prequential-all\request_profiles_validation
   OK c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\4_Feature_Engineering\artifacts\prequential-all\labels_train
   OK c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group

## Step 4: Load and Prepare Training and Validation Data

Load the complete training and validation datasets, including user click histories, impression timestamps, and contextual features. Perform data-quality checks to ensure that the user profiles are correctly aligned with the corresponding Stage 1 impressions.

Prepare each user's historical interactions by converting them into fixed-length sequences, retaining up to 50 historical clicks and 10 recent clicks. Additionally, construct contextual features such as the time of day, time since the user's previous visit, history length, and number of interest groups.

Finally, organise the processed data into a structured format for subsequent feature scaling and Two-Tower model training, while verifying that the training and validation periods are chronologically separated.

In [ ]:
@dataclass
class Requests:
    split: str
    uid: np.ndarray        #impression_uid
    user: np.ndarray       #user code (used for data matching)
    ts: np.ndarray         #impression timestamp in nanoseconds
    hour: np.ndarray       #hour index
    off: np.ndarray        #starting and ending positions of each full click history
    flat: np.ndarray       # Flattened article indices from full click histories
    seq: np.ndarray        # Fixed-length historical click sequences
    recent: np.ndarray     # Fixed-length recent click sequences
    ctx: np.ndarray        

    def __len__(self):
        return len(self.uid)


# Simple: every visit of every user (train + validation) to measure "time since last visit".
visits = (pl.concat([pl.read_parquet(STAGE1 / f"impressions_{s}.parquet", columns=["user_code", "timestamp"])
                     for s in SPLITS_USED])
          .unique().sort(["user_code", "timestamp"]))
visit_gaps = visits.with_columns(
    gap_hours=(pl.col("timestamp") - pl.col("timestamp").shift(1).over("user_code")).dt.total_seconds() / 3600.0)

# Convert variable-length click histories into fixed-length sequences, applying masks and right padding
def history_matrix(rows_series, mask_series, width):
    off, flat = list_to_csr(rows_series)
    moff, mflat = list_to_csr(mask_series)
    if not np.array_equal(off, moff):
        raise ValueError("history_mask must align one-for-one with history_rows")
    flat = np.where(mflat.astype(bool), flat, 0).astype(np.int32)
    lengths = np.diff(off)
    longest = max(int(lengths.max()) if len(lengths) else 0, 1)
    dense = np.zeros((len(lengths), longest), dtype=np.int32)
    dense[np.repeat(np.arange(len(lengths)), lengths), np.arange(len(flat)) - np.repeat(off[:-1], lengths)] = flat
    order = np.argsort(dense == 0, axis=1, kind="stable")      # real articles first, empty slots last
    dense = np.take_along_axis(dense, order, axis=1)
    out = np.zeros((len(lengths), width), dtype=np.int32)
    keep = min(width, longest)
    out[:, :keep] = dense[:, :keep]
    return out

#load and prepare impression-level user information for the specified dataset split
def load_requests(split):
    path = REAL_PROFILES[split]
    columns = ["impression_uid", "user_code", "timestamp", "history_rows", "history_mask",
               "full_history_rows", "recent_rows", "history_length", "interest_group_count"]
    profiles = pl.read_parquet(parquet_files(path), columns=columns)
    stage1 = pl.read_parquet(STAGE1 / f"impressions_{split}.parquet",
                             columns=["impression_uid", "user_code", "timestamp"])

    check("requests", f"{split}: profile rows = Stage 1 impressions", profiles.height, stage1.height)
    check("requests", f"{split}: unique impression_uid", profiles["impression_uid"].n_unique(), profiles.height)
    frame = stage1.join(profiles.rename({"user_code": "p_user", "timestamp": "p_time"}),
                        on="impression_uid", how="left")
    check("requests", f"{split}: impressions without a profile", int(frame["p_time"].null_count()), 0)
    check("requests", f"{split}: profile timestamps match Stage 1",
          bool((frame["timestamp"] == frame["p_time"]).all()), True)
    check("requests", f"{split}: profile users match Stage 1", bool((frame["user_code"] == frame["p_user"]).all()), True)

    frame = frame.join(visit_gaps, on=["user_code", "timestamp"], how="left").sort(["timestamp", "impression_uid"])

    off, flat = list_to_csr(frame["full_history_rows"])
    flat = flat.astype(np.int32)
    check("requests", f"{split}: history rows inside the article bank",
          bool(flat.size == 0 or (flat.min() >= 0 and flat.max() < N_ROWS)), True)
    mismatch = int((np.diff(off) != frame["history_length"].to_numpy()).sum())
    check("requests", f"{split}: history_length differs from len(full_history_rows)", mismatch,
          status="INFO", detail="history_length is used only as a context number")

    seq = history_matrix(frame["history_rows"], frame["history_mask"], CFG.MAX_HISTORY)
    recent = padded_tail(frame["recent_rows"], CFG.MAX_RECENT)

    ts = frame.select(pl.col("timestamp").dt.epoch("ns"))["timestamp"].to_numpy().astype(np.int64)
    hour = (ts - T0_NS) // HOUR_NS
    check("requests", f"{split}: timestamps inside 9-14 Nov 2019",
          bool(hour.min() >= 0 and hour.max() < N_HOURS), True)
    gap = frame["gap_hours"].to_numpy()
    hour_of_day = (hour % 24).astype(np.float32)
    ctx = np.stack([
        np.sin(2 * np.pi * hour_of_day / 24),
        np.cos(2 * np.pi * hour_of_day / 24),
        np.log1p(np.nan_to_num(gap, nan=0.0)),
        np.isnan(gap),
        np.log1p(frame["history_length"].fill_null(0).to_numpy().clip(0)),
        np.log1p(frame["interest_group_count"].fill_null(0).to_numpy().clip(0)),
    ], axis=1).astype(np.float32)

    return Requests(split, frame["impression_uid"].to_numpy().astype(np.int64),
                    frame["user_code"].to_numpy().astype(np.int64), ts, hour.astype(np.int64),
                    off, flat, seq, recent, ctx)


TRAIN = load_requests("train")
VALID = load_requests("validation")

for req in (TRAIN, VALID):
    lengths = np.diff(req.off)
    print(f"{req.split}: {len(req):,} requests | mean full history {lengths.mean():.1f} | "
          f"empty history {int((lengths == 0).sum()):,} | with recent clicks {int((req.recent > 0).any(1).sum()):,}")

check('time', 'training impressions precede validation',
      bool(TRAIN.ts.max() <= VALID.ts.min()), True)


train: 1,897,835 requests | mean full history 36.3 | empty history 11,837 | with recent clicks 918,020
validation: 334,913 requests | mean full history 40.3 | empty history 1,021 | with recent clicks 181,944


True

## Step 5: Construct User Profiles and Standardise Features

Construct user preference profiles by aggregating the text and entity embeddings of previously clicked articles, along with the distribution of article categories in each user's history. These features capture the user's general interests based on historical interactions.

Calculate the mean and standard deviation of the user profile and contextual features using only the training dataset, then apply the same standardisation to both training and validation data to prevent data leakage.

Finally, combine the historical click sequences, recent clicks, aggregated user profiles, category preferences, and contextual features into batches for processing by the User Tower.

In [ ]:
#construct user preference profiles by aggregating historical article embeddings and category distributions
@torch.no_grad()
def raw_profile(requests, idx):
    #retrieve the complete click histories for the selected impressions.
    rows, owner, lengths = csr_gather(requests.off, requests.flat, idx)
    n = len(idx)
    rows_t = torch.from_numpy(rows).to(DEVICE)
    owner_t = torch.from_numpy(owner).to(DEVICE)
    #aggregate text and entity embeddings for each user and apply L2 normalisation
    parts = []
    for matrix in (TEXT_T, ENTITY_T):
        vectors = matrix[rows_t]
        valid = vectors.abs().sum(1) > 0                     
        sums = torch.zeros(n, matrix.shape[1], device=DEVICE)
        sums.index_add_(0, owner_t[valid], vectors[valid])
        parts.append(F.normalize(sums, dim=1))
    counts = np.bincount(owner * N_CAT + cat_code[rows], minlength=n * N_CAT)
    interest = (counts.reshape(n, N_CAT) / np.maximum(lengths, 1)[:, None]).astype(np.float32)
    return torch.cat(parts, dim=1), torch.from_numpy(interest).to(DEVICE), rows, owner


#calculate the mean and standard deviation of user profiles across the training set
total = torch.zeros(164, dtype=torch.float64, device=DEVICE)
squares = torch.zeros(164, dtype=torch.float64, device=DEVICE)
for start in range(0, len(TRAIN), 4_096):
    profile, _, _, _ = raw_profile(TRAIN, np.arange(start, min(start + 4_096, len(TRAIN))))
    total += profile.double().sum(0)
    squares += (profile.double() ** 2).sum(0)
PROFILE_MEAN = (total / len(TRAIN)).float()
PROFILE_STD = torch.sqrt(torch.clamp(squares / len(TRAIN) - (total / len(TRAIN)) ** 2, min=0)).float()
PROFILE_STD[PROFILE_STD < 1e-6] = 1.0

#calculate the contextual feature mean and standard deviation across the training set and apply normalization
CONTEXT_MEAN = TRAIN.ctx.mean(0)
CONTEXT_STD = TRAIN.ctx.std(0)
CONTEXT_STD[CONTEXT_STD < 1e-6] = 1.0

#apply training-derived standardisation to both training and validation
for req in (TRAIN, VALID):
    req.ctx = ((req.ctx - CONTEXT_MEAN) / CONTEXT_STD).astype(np.float32)
    check("scaling", f"{req.split}: finite context values", bool(np.isfinite(req.ctx).all()), True)

USER_SCALER = {"profile_mean": PROFILE_MEAN.cpu().numpy(), "profile_std": PROFILE_STD.cpu().numpy(),
               "context_mean": CONTEXT_MEAN, "context_std": CONTEXT_STD}


def user_batch(requests, idx, model_cfg):
    idx = np.asarray(idx, dtype=np.int64)
    profile, interest, rows, owner = raw_profile(requests, idx)
    inputs = {
        "history": torch.from_numpy(requests.seq[idx].astype(np.int64)).to(DEVICE),
        "recent": torch.from_numpy(requests.recent[idx].astype(np.int64)).to(DEVICE),
        "profile": (profile - PROFILE_MEAN) / PROFILE_STD,
        "interest": interest,
        "context": torch.from_numpy(requests.ctx[idx]).to(DEVICE),
    }
    return inputs, rows, owner


example, _, _ = user_batch(TRAIN, np.arange(4), CFG)
print({k: tuple(v.shape) for k, v in example.items()})


{'history': (4, 50), 'recent': (4, 10), 'profile': (4, 164), 'interest': (4, 19), 'context': (4, 6)}


## Step 6: Prepare Training Targets and Negative Samples

Load the clicked articles from Stage 4 labels and verify their consistency with Stage 1 interaction records. Construct positive training examples from each clicked article while excluding articles already present in the user's click history to maintain consistency with the candidate retrieval process.

For each positive example, retrieve up to four exposed but unclicked articles as negative samples. Perform data-quality checks to ensure that these negatives do not overlap with clicked articles from the same impression.

Additionally, calculate hourly article click probabilities for Log-Q correction during training, helping account for differences in article sampling frequencies. Finally, organise the positive examples and negative samples for use in the model's loss function.

In [ ]:
#Create a mapping between impression IDs and their positions in the request arrays
def request_index(requests):
    return pl.DataFrame({"impression_uid": requests.uid, "req": np.arange(len(requests), dtype=np.int64)})

# Convert article identifiers into embedding row indices
def to_rows(codes):
    codes = np.asarray(codes, dtype=np.int64)
    if codes.size and (codes.max() >= len(row_of_code) or (row_of_code[codes] == 0).any()):
        raise ValueError("Labels reference articles outside the Stage 3 article bank")
    return row_of_code[codes]

#retrieve clicked articles for each impression
def clicks_by_request(requests, split):
    clicks = (pl.read_parquet(parquet_files(REAL_LABELS[split]), columns=["impression_uid", "news_code", "clicked"])
              .filter(pl.col("clicked") == 1)
              .join(request_index(requests), on="impression_uid", how="inner")
              .sort("req"))
    stage1_clicks = (pl.scan_parquet(STAGE1 / f"candidates_{split}.parquet")
                     .filter(pl.col("clicked") == 1).select(pl.len()).collect().item())
    check("labels", f"{split}: label clicks = Stage 1 clicks", clicks.height, stage1_clicks)
    req = clicks["req"].to_numpy()
    off = np.zeros(len(requests) + 1, dtype=np.int64)
    np.cumsum(np.bincount(req, minlength=len(requests)), out=off[1:])
    return off, to_rows(clicks["news_code"].to_numpy()).astype(np.int32), clicks

# Load clicked articles from the training and validation datasets
TRAIN_CLICK_OFF, TRAIN_CLICK_FLAT, train_clicks = clicks_by_request(TRAIN, "train")
VALID_CLICK_OFF, VALID_CLICK_FLAT, _ = clicks_by_request(VALID, "validation")

pos_req = train_clicks["req"].to_numpy().astype(np.int64)
pos_row = to_rows(train_clicks["news_code"].to_numpy())
pos_code = train_clicks["news_code"].to_numpy().astype(np.int64)

# Exclude positive articles already present in the user's history to match the retrieval policy
owner_all = np.repeat(np.arange(len(TRAIN), dtype=np.int64), np.diff(TRAIN.off))
in_history = np.isin(pos_req * N_ROWS + pos_row, owner_all * N_ROWS + TRAIN.flat.astype(np.int64))
del owner_all
check("training", "clicks dropped because already in history", int(in_history.sum()), status="INFO",
      detail="serving excludes history, same as the content baseline")
pos_req, pos_row, pos_code = pos_req[~in_history], pos_row[~in_history], pos_code[~in_history]
pos_hour = TRAIN.hour[pos_req]

# Load exposed but unclicked articles associated with each positive training example
groups = (pl.read_parquet(parquet_files(REAL_NEGATIVE_GROUPS), columns=["impression_uid", "positive_news_code", "negative_news_codes"])
          .join(request_index(TRAIN), on="impression_uid", how="inner")
          .select("req", pl.col("positive_news_code").cast(pl.Int64), "negative_news_codes"))
check("negatives", "duplicate (impression, positive) groups",
      groups.height - groups.select("req", "positive_news_code").unique().height, 0)
example_keys = pl.DataFrame({"req": pos_req, "positive_news_code": pos_code, "example": np.arange(len(pos_req))})
attached = example_keys.join(groups, on=["req", "positive_news_code"], how="left").sort("example")
# Retain up to four exposed negatives per positive example and pad missing positions with zeros
SLATE = padded_tail(attached["negative_news_codes"].list.head(CFG.NEGATIVES_PER_POS),
                    CFG.NEGATIVES_PER_POS).astype(np.int64)
SLATE[SLATE > 0] = to_rows(SLATE[SLATE > 0])
check("negatives", "clicks with at least one slate negative (share)",
      round(float((SLATE > 0).any(1).mean()), 4), status="INFO")

# Verify that negative samples do not include articles clicked within the same impression
click_keys = np.repeat(np.arange(len(TRAIN), dtype=np.int64), np.diff(TRAIN_CLICK_OFF)) * N_ROWS + TRAIN_CLICK_FLAT
slate_keys = (np.repeat(pos_req, CFG.NEGATIVES_PER_POS) * N_ROWS + SLATE.reshape(-1))[SLATE.reshape(-1) > 0]
check("negatives", "slate negatives that were actually clicked", int(np.isin(slate_keys, click_keys).sum()), 0)

key = pos_hour * N_ROWS + pos_row
_, inverse, counts = np.unique(key, return_inverse=True, return_counts=True)
per_hour = np.bincount(pos_hour, minlength=N_HOURS)
POS_LOGQ = np.log(counts[inverse] / per_hour[pos_hour]).astype(np.float32)

EXAMPLES = {"req": pos_req, "row": pos_row, "hour": pos_hour, "logq": POS_LOGQ, "slate": SLATE}
print(f"Training clicks used: {len(pos_req):,} | training hours: {len(np.unique(pos_hour))} | "
      f"validation clicks: {len(VALID_CLICK_FLAT):,}")


Training clicks used: 2,833,143 | training hours: 128 | validation clicks: 514,658


## Step 7: Construct Historical Hourly Article Features

Construct time-dependent article engagement features using historical exposure and click records from Stage 1. Aggregate interactions into hourly intervals and calculate cumulative statistics to capture changes in article popularity over time.

For each article, generate 10 features, including recent and cumulative exposure counts, recent click counts, smoothed CTR, article age, time since the last exposure, and indicators for newly observed or low-exposure articles.

To prevent data leakage, all engagement features are calculated using only interactions from previously completed hours. Training and validation follow a prequential approach, allowing earlier observed interactions to update the features used for subsequent recommendations.

Finally, standardise the numerical features using statistics calculated from training hours only and convert them into PyTorch tensors for the Article Tower.

In [ ]:
#aggregate article exposures and clicks by hour 
hourly = (pl.concat([pl.scan_parquet(STAGE1 / f"candidates_{s}.parquet")
                     .select("news_code", "clicked",
                             ((pl.col("timestamp").dt.epoch("ns") - T0_NS) // HOUR_NS).alias("h"),
                             pl.lit(s).alias("split"))
                     for s in SPLITS_USED])
          .group_by("news_code", "h", "split")
          .agg(exp=pl.len(), clk=pl.col("clicked").cast(pl.Int64).sum())
          .collect())
#convert hourly timestamps and article identifiers into numerical indices
h_idx = hourly["h"].to_numpy().astype(np.int64)
check("hourly", "log hours inside 9-14 Nov", bool(h_idx.min() >= 0 and h_idx.max() < N_HOURS), True)
r_idx = to_rows(hourly["news_code"].to_numpy())

#initialize and populate exposure and click matrices for each hour and article
EXPOSURES = np.zeros((N_HOURS, N_ROWS), dtype=np.int32)
CLICKS = np.zeros((N_HOURS, N_ROWS), dtype=np.int32)
np.add.at(EXPOSURES, (h_idx, r_idx), hourly["exp"].to_numpy())
np.add.at(CLICKS, (h_idx, r_idx), hourly["clk"].to_numpy())
stage1_rows = sum(pl.scan_parquet(STAGE1 / f"candidates_{s}.parquet").select(pl.len()).collect().item()
                  for s in SPLITS_USED)
check("hourly", "counted exposures = Stage 1 candidate rows", int(EXPOSURES.sum()), stage1_rows)

#Identify articles exposed at least once during training for diagnostics and potential cold-start analysis
train_part = hourly.filter(pl.col("split") == "train")
SHOWN_IN_TRAIN = np.zeros(N_ROWS, dtype=bool)
SHOWN_IN_TRAIN[to_rows(train_part["news_code"].to_numpy())] = True
del hourly, train_part, h_idx, r_idx

# Calculate cumulative exposure and click counts using only interactions from previous hours
CUM_EXP = np.zeros((N_HOURS + 1, N_ROWS), dtype=np.int32)
CUM_CLK = np.zeros((N_HOURS + 1, N_ROWS), dtype=np.int32)
np.cumsum(EXPOSURES, axis=0, out=CUM_EXP[1:])
np.cumsum(CLICKS, axis=0, out=CUM_CLK[1:])
TOTAL_EXP = CUM_EXP.sum(axis=1, dtype=np.int64)
TOTAL_CLK = CUM_CLK.sum(axis=1, dtype=np.int64)

# Calculate total historical exposures and clicks across all articles for each hourly cutoff
LAST_SEEN = np.full((N_HOURS, N_ROWS), -1, dtype=np.int16)
running = np.full(N_ROWS, -1, dtype=np.int16)
for h in range(N_HOURS):
    LAST_SEEN[h] = running
    running[EXPOSURES[h] > 0] = h

AVAIL_HOURS = (np.minimum(avail_ns, T0_NS + 10 * N_HOURS * HOUR_NS) - T0_NS) / HOUR_NS

#Construct 10 hourly engagement and availability features for every article using past interactions only
def snapshot(h):
    lo = max(h - 6, 0)
    seen = LAST_SEEN[h]
    never = seen < 0
    prior_ctr = (TOTAL_CLK[h] + 1) / (TOTAL_EXP[h] + 2)              # average CTR so far
    out = np.empty((N_ROWS, 10), dtype=np.float32)
    out[:, 0] = np.log1p(EXPOSURES[h - 1]) if h else 0.0              # shown last hour
    out[:, 1] = np.log1p(CUM_EXP[h] - CUM_EXP[lo])                    # shown last 6 hours
    out[:, 2] = np.log1p(CUM_CLK[h] - CUM_CLK[lo])                    # clicked last 6 hours
    out[:, 3] = np.log1p(CUM_EXP[h])                                  # shown in total
    out[:, 4] = (CUM_CLK[h] + CTR_STRENGTH * prior_ctr) / (CUM_EXP[h] + CTR_STRENGTH)
    out[:, 5] = np.log1p(np.clip(h - AVAIL_HOURS, 0, None))           # age in hours
    out[:, 6] = np.log1p(np.where(never, 0, h - seen.astype(np.int64)))  # hours since last shown
    out[:, 7] = never
    out[:, 8] = history_only
    out[:, 9] = CUM_EXP[h] < 100
    return out


#training-only mean and spread for features 0-6.
s1, s2, count = np.zeros(7), np.zeros(7), 0
for h in np.unique(TRAIN.hour):
    block = snapshot(int(h))[rows_by_avail[:eligible_count(int(h))], :7].astype(np.float64)
    s1 += block.sum(0)
    s2 += (block ** 2).sum(0)
    count += len(block)
HOURLY_MEAN = (s1 / count).astype(np.float32)
HOURLY_STD = np.sqrt(np.maximum(s2 / count - (s1 / count) ** 2, 1e-12)).astype(np.float32)


def snapshot_tensor(h):
    #standardised snapshot on the model's device.
    state = snapshot(h)
    state[:, :7] = (state[:, :7] - HOURLY_MEAN) / HOURLY_STD
    return torch.from_numpy(state).to(DEVICE)


print(f"Exposures counted: {int(EXPOSURES.sum()):,} | articles shown in train: {int(SHOWN_IN_TRAIN.sum()):,}")
print("Hourly feature means (0-6):", np.round(HOURLY_MEAN, 3))
print("Hourly feature spreads (0-6):", np.round(HOURLY_STD, 3))

check('hourly', 'first-hour previous exposure count is zero',
      bool(np.all(snapshot(0)[:, 0] == 0)), True)


Exposures counted: 83,507,374 | articles shown in train: 24,641
Hourly feature means (0-6): [0.098 0.193 0.053 0.678 0.041 3.698 0.438]
Hourly feature spreads (0-6): [0.612 0.931 0.408 1.816 0.023 0.997 1.115]


True

## Step 8: Shared Article Content Encoder

Implement a shared neural network encoder to transform article content and metadata into 128-dimensional embeddings. The encoder combines precomputed text and entity embeddings from Stage 3 with trainable category and subcategory embeddings, along with an indicator for missing entity information.

The combined features are passed through fully connected layers with layer normalisation, GELU activation, and dropout to learn meaningful article representations while reducing overfitting.

The same encoder is shared between the User Tower and Article Tower, ensuring consistent article representations across both components. By learning from article content rather than relying solely on historical interactions, the model can also generate representations for newly available articles with limited engagement history, helping address the article cold-start problem.

In [ ]:
class NewsEncoder(nn.Module):
    """Generate 128-dimensional article embeddings shared by the User Tower and Article Tower."""

    def __init__(self, cfg):
        super().__init__()
        d = cfg.EMBED_DIM
        #Store the precomputed article features from Stage 3 without updating them during training
        self.register_buffer("text64", TEXT_T, persistent=False)
        self.register_buffer("entity100", ENTITY_T, persistent=False)
        self.register_buffer("cat", CAT_T, persistent=False)
        self.register_buffer("subcat", SUB_T, persistent=False)
        #Learn how to transform the text and entity embeddings
        self.text_proj = nn.Sequential(nn.Linear(64, 64), nn.LayerNorm(64), nn.GELU())
        self.entity_proj = nn.Sequential(nn.Linear(100, 32), nn.LayerNorm(32), nn.GELU())
        self.cat_embed = nn.Embedding(N_CAT, 16)
        self.subcat_embed = nn.Embedding(N_SUB, 8)
        #Combine all article features and learn a final 128-dimensional embedding
        self.network = nn.Sequential(
            nn.Linear(64 + 32 + 1 + 16 + 8, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(cfg.ITEM_DROPOUT),
            nn.Linear(256, d))

    def forward(self, article_rows):
        entity = self.entity100[article_rows]
        has_entity = (entity.abs().sum(-1, keepdim=True) > 0).float() 
        # Combine the article's text, entity, category, and subcategory features
        features = torch.cat([
            self.text_proj(self.text64[article_rows]),
            self.entity_proj(entity),
            has_entity,
            self.cat_embed(self.cat[article_rows]),
            self.subcat_embed(self.subcat[article_rows]),
        ], dim=-1)
        # Generate the article embeddings and keep padding positions as zeros
        out = self.network(features)
        return out.masked_fill((article_rows == 0).unsqueeze(-1), 0.0)


## Step 9: User Tower Architecture

Implement the User Tower to learn 128-dimensional user embeddings based on historical interactions and contextual information. Previously clicked articles are processed through the shared News Encoder, followed by self-attention to capture relationships between articles and attention pooling to identify the most important historical interests.

A separate recent-history component captures short-term preferences by averaging the embeddings of recently clicked articles. For users with no recorded click history, learnable default embeddings are used to help address the user cold-start problem.

Finally, combine the historical and recent preferences with aggregated user profiles, category interests, and contextual features. The combined representation is passed through fully connected layers with layer normalisation, GELU activation, and dropout to produce a normalised user embedding for candidate retrieval.

In [ ]:
class UserTower(nn.Module):
    def __init__(self, news_encoder, cfg):
        super().__init__()
        d = cfg.EMBED_DIM
        self.cfg = cfg
        self.news = news_encoder
        #Apply self-attention to learn relationships between historical clicks
        self.self_attn = nn.MultiheadAttention(d, num_heads=4, dropout=0.1, batch_first=True)
        self.history_norm = nn.LayerNorm(d)
        #Learn which historical articles are more important to the user's interests
        self.attention_score = nn.Sequential(nn.Linear(d, d), nn.Tanh(), nn.Linear(d, 1))
        #Learn default representations for users with missing historical or recent clicks
        self.no_hist = nn.Parameter(torch.zeros(d))
        self.no_recent = nn.Parameter(torch.zeros(d))
        # Combine user features through fully connected layers to produce a 128-dimensional user embedding
        self.network = nn.Sequential(
            nn.Linear(2 * d + 164 + N_CAT + 6, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(cfg.USER_DROPOUT),
            nn.Linear(512, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(cfg.USER_DROPOUT),
            nn.Linear(256, d))

    def pool(self, encoded, mask, attention):
         # Combine article embeddings using either learned attention weights or simple averaging
        if attention:
            logits = self.attention_score(encoded).squeeze(-1).masked_fill(~mask, -1e4)
            weights = torch.softmax(logits, dim=1) * mask.float()
        else:
            weights = mask.float()
        # Normalise the weights and exclude padded articles from the calculation
        weights = weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        return (weights.unsqueeze(-1) * encoded).sum(dim=1)

    def forward(self, batch):
        history = batch["history"]
        if self.training:
            # Optionally remove historical clicks during training to improve robustness to sparse or missing user histories
            hide_item = torch.rand(history.shape, device=history.device) < HISTORY_ITEM_DROPOUT
            hide_all = torch.rand(history.shape[0], 1, device=history.device) < HISTORY_DROPOUT
            history = history.masked_fill(hide_item | hide_all, 0)
        mask = history != 0
        has_history = mask.any(dim=1, keepdim=True)
        encoded = self.news(history)                                 

        # Use self-attention to capture relationships between previously clicked articles
        if self.cfg.USE_SELF_ATTENTION:                               
            pad = ~mask.clone()
            pad[~mask.any(dim=1), 0] = False                          
            attended, _ = self.self_attn(encoded, encoded, encoded, key_padding_mask=pad, need_weights=False)
            encoded = self.history_norm(encoded + attended).masked_fill(~mask.unsqueeze(-1), 0.0)

        pooled = self.pool(encoded, mask, self.cfg.USE_ATTENTION_POOLING)  
        pooled = torch.where(has_history, pooled, self.no_hist[None, :])

        # Average recent article embeddings to capture the user's short-term interests
        if self.cfg.USE_RECENT:                                       
            recent_mask = batch["recent"] != 0
            recent = self.pool(self.news(batch["recent"]), recent_mask, attention=False)
            recent = torch.where(recent_mask.any(dim=1, keepdim=True), recent, self.no_recent[None, :])
        else:
            recent = torch.zeros_like(pooled)

        # Combine historical preferences, recent interests, user profiles, category interests, and context
        joined = torch.cat([pooled, recent, batch["profile"], batch["interest"], batch["context"]], dim=-1) 
        return F.normalize(self.network(joined), dim=-1)              


## Step 10: Article Tower and Two-Tower Model Architecture

Implement the Article Tower to generate 128-dimensional article embeddings by combining content representations from the shared News Encoder with historical hourly engagement features. The 10 hourly features are projected into a 32-dimensional representation before being combined with article content and passed through fully connected layers with layer normalisation, GELU activation, and dropout.

Next, construct the complete Two-Tower model by integrating the User Tower and Article Tower, both of which share the same News Encoder. The resulting user and article embeddings are L2-normalised, allowing their similarity to be calculated using the dot product for candidate retrieval.

Additionally, introduce a learnable temperature parameter to control the sensitivity of similarity scores during contrastive training. Finally, initialise the model and display the total number of trainable parameters.

In [ ]:
class ItemTower(nn.Module):
    def __init__(self, news_encoder, cfg):
        super().__init__()
        self.cfg = cfg
        # Use the shared News Encoder to generate article content embeddings
        self.news = news_encoder                                      
        self.hourly_proj = nn.Sequential(nn.Linear(10, 32), nn.GELU())
        # Combine article content and hourly features to generate a 128-dimensional embedding
        self.network = nn.Sequential(
            nn.Linear(cfg.EMBED_DIM + 32, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(cfg.ITEM_DROPOUT),
            nn.Linear(256, cfg.EMBED_DIM))

    def forward(self, article_rows, hourly):
        #Generate article embeddings using text, entities, and categorical information
        content = self.news(article_rows)                           
        if self.cfg.USE_HOURLY:
            state = self.hourly_proj(hourly)                       
        else:
            state = torch.zeros((*content.shape[:-1], 32), device=content.device)
        return F.normalize(self.network(torch.cat([content, state], dim=-1)), dim=-1)


class TwoTowerRetriever(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # Create one shared News Encoder for both the User Tower and Article Tower
        self.news_encoder = NewsEncoder(cfg)                       .
        self.user = UserTower(self.news_encoder, cfg)               
        self.item = ItemTower(self.news_encoder, cfg)                
        
        self.log_temperature = nn.Parameter(torch.tensor(math.log(cfg.INIT_TEMPERATURE)),
                                            requires_grad=cfg.LEARN_TEMPERATURE)

    @property
    def temperature(self):
        return self.log_temperature.exp().clamp(*TEMPERATURE_RANGE)

# Initialise the Two-Tower model and display the number of trainable parameters
probe = TwoTowerRetriever(CFG).to(DEVICE)
print("Learned parameters:", f"{sum(p.numel() for p in probe.parameters() if p.requires_grad):,}")
del probe


Learned parameters: 627,090


## Step 11: Negative Sampling and Loss Function

Prepare training batches by grouping positive interactions within the same hour to ensure consistent historical article features. Construct negative samples using three sources: exposed but unclicked articles from the original impressions, in-batch negatives from other positive interactions, and randomly sampled articles from the time-eligible catalogue.

Apply masking to prevent invalid comparisons, including previously clicked articles, other positive articles from the same impression, and articles unavailable at the prediction time. Log-Q correction is also applied to account for differences in article sampling probabilities.

Finally, calculate a temperature-scaled softmax cross-entropy loss to train the model to assign higher similarity scores to clicked articles than competing candidates. This encourages the Two-Tower model to retrieve relevant articles more effectively during candidate generation.

In [ ]:
#Group training clicks by hour and shuffle batches during training
def hour_batches(rng, batch_size):
    order = np.argsort(EXAMPLES["hour"], kind="stable")
    hours, starts = np.unique(EXAMPLES["hour"][order], return_index=True)
    ends = np.r_[starts[1:], len(order)]
    for k in rng.permutation(len(hours)):
        ids = order[starts[k]:ends[k]].copy()
        rng.shuffle(ids)
        for s in range(0, len(ids), batch_size):
            batch = ids[s:s + batch_size]
            if len(batch) >= 1:
                yield int(hours[k]), batch

#Identify articles that should not be treated as negative examples
def lineup_masks(batch, random_rows):
    req, pos = EXAMPLES["req"][batch], EXAMPLES["row"][batch]
    n, n_random = len(batch), len(random_rows)
    columns_of = {}
    for column, article_row in enumerate(pos.tolist()):
        columns_of.setdefault(article_row, []).append(column)
    random_column = {article_row: column for column, article_row in enumerate(random_rows.tolist())}
    hide_inbatch = np.zeros((n, n), dtype=bool)
    hide_random = np.zeros((n, n_random), dtype=bool)
    # Prevent articles already clicked by the user from being treated as negative samples
    for r, q in enumerate(req.tolist()):
        blocked = set(TRAIN.flat[TRAIN.off[q]:TRAIN.off[q + 1]].tolist())                    
        blocked.update(TRAIN_CLICK_FLAT[TRAIN_CLICK_OFF[q]:TRAIN_CLICK_OFF[q + 1]].tolist())  
        blocked.add(int(pos[r]))                                                              
        for article_row in blocked:
            for column in columns_of.get(article_row, ()):
                if column != r:
                    hide_inbatch[r, column] = True
            column = random_column.get(article_row)
            if column is not None:
                hide_random[r, column] = True
    request_time = TRAIN.ts[req]
    # Exclude articles that were not yet available when the impression occurred
    hide_inbatch |= avail_ns[pos][None, :] > request_time[:, None]          
    np.fill_diagonal(hide_inbatch, False)                                    
    hide_random |= avail_ns[random_rows][None, :] > request_time[:, None]
    return torch.from_numpy(hide_inbatch).to(DEVICE), torch.from_numpy(hide_random).to(DEVICE)


def lineup_loss(model, cfg, hour, batch, state, rng):

    n = len(batch)
    tau = model.temperature
    #Randomly sample negative articles from the catalogue available during this hour
    n_eligible = eligible_count(hour)
    n_random = min(cfg.UNIFORM_NEGATIVES, n_eligible)
    random_rows = rows_by_avail[rng.choice(n_eligible, size=n_random, replace=False)] if n_random else np.zeros(0, np.int64)
    hide_inbatch, hide_random = lineup_masks(batch, random_rows)

    # Generate user embeddings for the current batch
    inputs, _, _ = user_batch(TRAIN, EXAMPLES["req"][batch], cfg)
    u = model.user(inputs)                                                 

    # Generate embeddings for the articles that users actually clicked
    pos_t = torch.from_numpy(EXAMPLES["row"][batch]).to(DEVICE)
    v_pos = model.item(pos_t, state[pos_t])

    # Generate embeddings for exposed but unclicked articles                                   
    slate_t = torch.from_numpy(EXAMPLES["slate"][batch]).to(DEVICE)
    v_slate = model.item(slate_t.reshape(-1), state[slate_t.reshape(-1)]).view(n, cfg.NEGATIVES_PER_POS, -1)

    pieces = []
    if cfg.USE_INBATCH_NEGATIVES:
        
        s_inbatch = (u @ v_pos.T) / tau
        if USE_LOGQ:
            s_inbatch = s_inbatch - torch.from_numpy(EXAMPLES["logq"][batch]).to(DEVICE)[None, :]
        pieces.append(s_inbatch.masked_fill(hide_inbatch, float("-inf")))
        target = torch.arange(n, device=DEVICE)
    else:
        pieces.append(((u * v_pos).sum(-1, keepdim=True)) / tau)             
        target = torch.zeros(n, dtype=torch.long, device=DEVICE)

    s_slate = torch.einsum("bd,bkd->bk", u, v_slate) / tau                   
    pieces.append(s_slate.masked_fill(slate_t == 0, float("-inf")))

    if n_random:
        random_t = torch.from_numpy(random_rows).to(DEVICE)
        s_random = (u @ model.item(random_t, state[random_t]).T) / tau       
        if USE_LOGQ:
            s_random = s_random - math.log(n_random / n_eligible)
        pieces.append(s_random.masked_fill(hide_random, float("-inf")))

    return F.cross_entropy(torch.cat(pieces, dim=1), target)

print("Loss ready")


Loss ready


## 12 · Full-Catalogue Retrieval and Recall@M

Evaluate the model by ranking all articles available at each impression's timestamp, using historical features from previous hours.

Exclude previously clicked articles, retrieve the Top-M recommendations, and compare them against actual clicks to calculate Recall@50, 100, 200, and 500.

In [ ]:
class ModelScorer:
    """Scores requests with the two-tower model. Item vectors are rebuilt once per hour."""

    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def start_hour(self, h, eligible_t):
        # Generate embeddings for all articles available during the respective hour
        state = snapshot_tensor(h)
        self.item_vectors = torch.cat([self.model.item(eligible_t[s:s + 65536], state[eligible_t[s:s + 65536]])
                                       for s in range(0, len(eligible_t), 65536)])

    @torch.no_grad()
    def scores(self, requests, idx):
        # Compare user embeddings with article embeddings to get recommendation scores
        inputs, rows, owner = user_batch(requests, idx, self.model.cfg)
        return self.model.user(inputs) @ self.item_vectors.T, rows, owner


class RecentPopularityScorer:
    """Recommend articles based on how popular they were in previous hours."""

    def start_hour(self, h, eligible_t):
        #Give more importance to last hour's exposures, followed by total exposures
        state = snapshot(h)
        score = state[:, 0] * 1000.0 + state[:, 3]                     
        self.score = torch.from_numpy(score.astype(np.float32)).to(DEVICE)[eligible_t]

    def scores(self, requests, idx):
        rows, owner, _ = csr_gather(requests.off, requests.flat, idx)
        return self.score[None, :].expand(len(idx), -1).clone(), rows, owner


@torch.no_grad()
def retrieve(scorer, requests, k=500):
    """Find the Top-K recommended articles for each user request"""
    
    if hasattr(scorer, "model"):
        scorer.model.eval()
    column_of_row = np.full(N_ROWS, -1, dtype=np.int64)
    order = np.argsort(requests.ts, kind="stable")
    hours = requests.hour[order]
    for h in np.unique(hours):
        selected = order[hours == h]
        #Find articles available by the end of the current hour
        n_eligible = eligible_count(int(h))
        if n_eligible < k:
            raise ValueError(f"Only {n_eligible} eligible articles in hour {h}")
        eligible = rows_by_avail[:n_eligible]
        eligible_t = torch.from_numpy(eligible).to(DEVICE)
        scorer.start_hour(int(h), eligible_t)                           
        eligible_avail = AVAIL_T[eligible_t]
        column_of_row[:] = -1
        column_of_row[eligible] = np.arange(n_eligible)
        for s in range(0, len(selected), SERVE_BATCH):
            idx = selected[s:s + SERVE_BATCH]
            #Calculate recommendation scores for each user
            scores, hist_rows, hist_owner = scorer.scores(requests, idx)
            #Remove articles that were not yet available when the user visited
            request_time = torch.from_numpy(requests.ts[idx]).to(DEVICE)
            scores.masked_fill_(eligible_avail[None, :] > request_time[:, None], float("-inf"))  
            cols = column_of_row[hist_rows]
            keep = cols >= 0
            if keep.any():                                               
                scores[torch.from_numpy(hist_owner[keep]).to(DEVICE),
                       torch.from_numpy(cols[keep]).to(DEVICE)] = float("-inf")
            #Select the K articles with the highest scores
            top_scores, top_cols = scores.topk(k, dim=1)                  
            if not torch.isfinite(top_scores).all():
                raise ValueError(f"Fewer than {k} eligible, unread articles in hour {h}")
            yield idx, eligible[top_cols.cpu().numpy()]


def click_ranks(scorer, requests, click_off, click_flat, k=500, writer=None):
    """Check whether the recommended articles match what users actually clicked."""
    ranks = np.zeros(len(click_flat), dtype=np.int16)
    cutoffs = np.arange(1, k + 1, dtype=np.int16)
    for idx, top_rows in retrieve(scorer, requests, k):
        for j, q in enumerate(idx.tolist()):
            lo, hi = click_off[q], click_off[q + 1]
            if hi > lo:
                #Check which clicked articles appear in the Top-K recommendations.
                hits = top_rows[j][None, :] == click_flat[lo:hi, None]
                # Record the ranking position of each clicked article.
                ranks[lo:hi] = np.where(hits.any(1), hits.argmax(1) + 1, 0)
        if writer is not None:
            writer.add(np.repeat(requests.uid[idx], k), np.tile(cutoffs, len(idx)), row_to_code[top_rows].reshape(-1))
    return ranks


def recall_table(ranks, click_off, ms):
    """Calculate Recall@M and Hit Rate to evaluate recommendation performance."""
    owner = np.repeat(np.arange(len(click_off) - 1), np.diff(click_off))
    relevant = np.diff(click_off)
    has_click = relevant > 0
    out = []
    for m in ms:
        hit = (ranks > 0) & (ranks <= m)
        hits = np.bincount(owner, weights=hit, minlength=len(relevant))
        out.append({"m": m,
                    "recall_impression_mean": float((hits[has_click] / relevant[has_click]).mean()),
                    "recall_pooled": float(hit.sum() / len(ranks)),
                    "hit_rate": float((hits > 0).mean())})
    return pd.DataFrame(out)


def find_shared_evaluator(repo_root):
    """use the same evaluation code as the baselines to ensure consistent metrics"""
    
    preferred = repo_root / "5_Models" / "5_01_Candidate_Generation" / "cg_evaluation.py"
    if preferred.is_file():
        path = preferred
    else:
        matches = [p for p in repo_root.rglob("cg_evaluation.py") if ".ipynb_checkpoints" not in p.parts]
        if len(matches) != 1:
            raise FileNotFoundError(f"Expected one cg_evaluation.py next to the baselines, found: {matches}")
        path = matches[0]
    spec = importlib.util.spec_from_file_location("cg_evaluation", path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return path, module.CandidateEvaluator

# Use the same evaluator as the baseline models for a fair comparison.
EVALUATOR_PATH, CandidateEvaluator = find_shared_evaluator(REPO_ROOT)
evaluator = CandidateEvaluator(root=REPO_ROOT, split="validation")   
check("evaluation", "Stage 1 validation impressions = our requests", evaluator.impressions.height, len(VALID))
print("Using the baselines' evaluator:", EVALUATOR_PATH)


Using the baselines' evaluator: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\cg_evaluation.py


## 13 · Train the default model with automatic resume

Train for at most 20 epochs with early-stopping patience 3. Print loss and full-validation Recall@500 after every epoch, save an atomic checkpoint, and continue after the last completed epoch if interrupted.


In [ ]:
DEFAULT_RESUME = DEFAULT_DIR / 'resume_latest.pt'
DEFAULT_HISTORY = DEFAULT_DIR / 'training_history.csv'

def current_data_identity():
    paths = [STAGE3 / 'text64.npy', STAGE3 / 'article_entity100.npy',
             REAL_ARTICLES, REAL_AVAILABILITY]
    paths += [STAGE1 / f'{kind}_{split}.parquet'
              for kind in ('impressions', 'candidates') for split in SPLITS_USED]
    for split in SPLITS_USED:
        paths.extend(parquet_files(REAL_PROFILES[split]))
        paths.extend(parquet_files(REAL_LABELS[split]))
    paths.extend(parquet_files(REAL_NEGATIVE_GROUPS))
    return [(path.resolve().relative_to(REPO_ROOT.resolve()).as_posix(), path.stat().st_size)
            for path in sorted(set(paths))]

def clone_weights(state):
    return {k: v.detach().cpu().clone() for k, v in state.items()}

def atomic_checkpoint(path, payload):
    temporary = path.with_name(path.name + '.incomplete')
    torch.save(payload, temporary)
    temporary.replace(path)

def train_default_with_resume(cfg):
    signature = current_data_identity()
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(SEED)
    rng = np.random.default_rng(SEED)
    model = TwoTowerRetriever(cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR,
                                  weight_decay=cfg.WEIGHT_DECAY)
    best_score = -float('inf')
    best_epoch = 0
    best_weights = None
    no_improvement = 0
    completed = 0
    history = []

    if DEFAULT_RESUME.exists():
        saved = torch.load(DEFAULT_RESUME, map_location='cpu', weights_only=False)
        if (saved['version'] != 1 or saved['config'] != asdict(cfg) or
                saved['seed'] != SEED or saved['source_identity'] != signature or
                saved['device'] != DEVICE.type):
            raise ValueError('Existing hourly checkpoint has different data/config/device. '
                             'Use the original setup or move the checkpoint aside first.')
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        for values in optimizer.state.values():
            for key, value in values.items():
                if torch.is_tensor(value):
                    values[key] = value.to(DEVICE)
        best_score = float(saved['best_score'])
        best_epoch = int(saved['best_epoch'])
        best_weights = saved['best_weights']
        no_improvement = int(saved['no_improvement'])
        completed = int(saved['completed_epoch'])
        history = saved['history']
        rng.bit_generator.state = saved['numpy_generator_state']
        random.setstate(saved['python_rng_state'])
        np.random.set_state(saved['numpy_global_state'])
        torch.set_rng_state(saved['torch_rng_state'])
        if DEVICE.type == 'cuda':
            torch.cuda.set_rng_state_all(saved['cuda_rng_state'])
        pd.DataFrame(history).to_csv(DEFAULT_HISTORY, index=False)
        print(f'Resumed after epoch {completed}; best epoch {best_epoch} '
              f'Recall@500={best_score:.5f}', flush=True)
    else:
        print('Starting default hourly model from epoch 1', flush=True)

    if (completed >= cfg.FINAL_MAX_EPOCHS or
            no_improvement >= cfg.EARLY_STOP_PATIENCE):
        print('Training is already complete. Restoring best weights.', flush=True)

    for epoch in range(completed + 1, cfg.FINAL_MAX_EPOCHS + 1):
        if no_improvement >= cfg.EARLY_STOP_PATIENCE:
            break
        started = time.perf_counter()
        model.train()
        weighted_loss = 0.0
        positive_count = 0
        active_hour, hourly_tensor = None, None
        for hour, indices in hour_batches(rng, cfg.BATCH_SIZE):
            if hour != active_hour:
                active_hour = hour
                hourly_tensor = snapshot_tensor(hour)
            loss = lineup_loss(model, cfg, hour, indices, hourly_tensor, rng)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite training loss at epoch {epoch}')
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
            positive_count += len(indices)
            weighted_loss += float(loss.detach()) * len(indices)

        if positive_count == 0:
            raise ValueError('No eligible positive targets during training')
        if positive_count != len(EXAMPLES['req']):
            raise ValueError(f'Epoch {epoch} used {positive_count:,} of '
                             f'{len(EXAMPLES["req"]):,} eligible positive targets')
        valid_ranks = click_ranks(ModelScorer(model), VALID,
                                  VALID_CLICK_OFF, VALID_CLICK_FLAT, k=max(cfg.TOP_MS))
        metrics = recall_table(valid_ranks, VALID_CLICK_OFF, cfg.TOP_MS)
        score = float(metrics.loc[metrics.m == cfg.SELECTION_M,
                                  'recall_impression_mean'].iloc[0])
        improved = score > best_score + 1e-6
        if improved:
            best_score, best_epoch, no_improvement = score, epoch, 0
            best_weights = clone_weights(model.state_dict())
        else:
            no_improvement += 1
        average_loss = weighted_loss / positive_count
        seconds = time.perf_counter() - started
        history.append({'epoch': epoch, 'train_loss': average_loss,
                        'train_positive_examples': positive_count,
                        'validation_recall_at_500': score,
                        'best_recall_at_500': best_score,
                        'best_epoch': best_epoch,
                        'non_improving_epochs': no_improvement,
                        'seconds': seconds,
                        **{f'validation_recall_at_{m}': float(metrics.loc[
                            metrics.m == m, 'recall_impression_mean'].iloc[0])
                           for m in cfg.TOP_MS}})
        atomic_checkpoint(DEFAULT_RESUME, {
            'version': 1, 'config': asdict(cfg), 'seed': SEED,
            'source_identity': signature, 'device': DEVICE.type,
            'completed_epoch': epoch, 'model': clone_weights(model.state_dict()),
            'optimizer': optimizer.state_dict(), 'best_weights': best_weights,
            'best_score': best_score, 'best_epoch': best_epoch,
            'no_improvement': no_improvement, 'history': history,
                'numpy_generator_state': rng.bit_generator.state,
                'python_rng_state': random.getstate(),
            'numpy_global_state': np.random.get_state(),
            'torch_rng_state': torch.get_rng_state(),
            'cuda_rng_state': torch.cuda.get_rng_state_all()
                if DEVICE.type == 'cuda' else None,
        })
        pd.DataFrame(history).to_csv(DEFAULT_HISTORY, index=False)
        print(f'default_hourly | epoch {epoch:02d}/{cfg.FINAL_MAX_EPOCHS} | '
              f'train loss={average_loss:.5f} | positives={positive_count:,} | '
              f'FULL val Recall@500={score:.5f} | {seconds:.1f}s | '
              f'{"BEST" if improved else f"patience {no_improvement}/{cfg.EARLY_STOP_PATIENCE}"} | '
              'checkpoint saved', flush=True)
        if no_improvement >= cfg.EARLY_STOP_PATIENCE:
            print(f'Early stopping; restoring epoch {best_epoch}', flush=True)

    if best_weights is None:
        raise RuntimeError('No trained checkpoint with valid Recall@500 was produced')
    model.load_state_dict(best_weights)
    model.eval()
    return model, {'best_epoch': best_epoch, 'best_validation_recall_at_500': best_score}, pd.DataFrame(history)

DEFAULT_MODEL, DEFAULT_FIT, TRAIN_HISTORY = train_default_with_resume(CFG)
display(TRAIN_HISTORY)
print('Best validation epoch:', DEFAULT_FIT['best_epoch'],
      '| Recall@500:', round(DEFAULT_FIT['best_validation_recall_at_500'], 5))


Starting default hourly model from epoch 1
default_hourly | epoch 01/20 | train loss=7.52881 | positives=2,833,143 | FULL val Recall@500=0.93443 | 12669.7s | BEST | checkpoint saved
default_hourly | epoch 02/20 | train loss=7.45711 | positives=2,833,143 | FULL val Recall@500=0.92561 | 6741.2s | patience 1/3 | checkpoint saved
default_hourly | epoch 03/20 | train loss=7.40841 | positives=2,833,143 | FULL val Recall@500=0.93536 | 11869.5s | BEST | checkpoint saved
default_hourly | epoch 04/20 | train loss=7.37303 | positives=2,833,143 | FULL val Recall@500=0.92913 | 9234.3s | patience 1/3 | checkpoint saved
default_hourly | epoch 05/20 | train loss=7.34882 | positives=2,833,143 | FULL val Recall@500=0.93852 | 4131.1s | BEST | checkpoint saved
default_hourly | epoch 06/20 | train loss=7.33191 | positives=2,833,143 | FULL val Recall@500=0.93286 | 4239.0s | patience 1/3 | checkpoint saved
default_hourly | epoch 07/20 | train loss=7.30448 | positives=2,833,143 | FULL val Recall@500=0.93552 |

,epoch,train_loss,train_positive_examples,validation_recall_at_500,best_recall_at_500,best_epoch,non_improving_epochs,seconds,validation_recall_at_50,validation_recall_at_100,validation_recall_at_200
0,1,7.528809,2833143,0.934431,0.934431,1,0,12669.668329,0.639850,0.754583,0.847163
1,2,7.457111,2833143,0.925612,0.934431,1,1,6741.200213,0.652100,0.761078,0.845720
2,3,7.408410,2833143,0.935363,0.935363,3,0,11869.540127,0.656822,0.766729,0.855037
3,4,7.373026,2833143,0.929130,0.935363,3,1,9234.302573,0.656107,0.768776,0.854573
4,5,7.348823,2833143,0.938521,0.938521,5,0,4131.055675,0.669584,0.779191,0.864689
5,6,7.331910,2833143,0.932859,0.938521,5,1,4238.995632,0.665610,0.775006,0.861382
6,7,7.304484,2833143,0.935518,0.938521,5,2,4239.988787,0.662279,0.775340,0.861888
7,8,7.291507,2833143,0.935729,0.938521,5,3,4195.959518,0.669922,0.780700,0.867069


Best validation epoch: 5 | Recall@500: 0.93852


## 14 · Save the default model and its validation results

Save the best model separately from the resume checkpoint, including its hyperparameters, training-only scaling statistics, and hourly scaling statistics.


In [14]:
MODEL_CHECKPOINT = DEFAULT_DIR / 'model.pt'
torch.save({
    'model_state': DEFAULT_MODEL.state_dict(),
    'config': asdict(CFG), 'seed': SEED,
    'best_epoch': DEFAULT_FIT['best_epoch'],
    'best_validation_recall_at_500': DEFAULT_FIT['best_validation_recall_at_500'],
    'user_scaler': USER_SCALER,
    'hourly_mean': HOURLY_MEAN, 'hourly_std': HOURLY_STD,
    'hour_origin_ns': T0_NS, 'hour_size_ns': HOUR_NS,
    'ctr_strength': CTR_STRENGTH,
}, MODEL_CHECKPOINT)
report_qc().to_csv(SHARED_DIR / 'quality_checks.csv', index=False)
print('Saved best default hourly model:', MODEL_CHECKPOINT)
print('Resume checkpoint:', DEFAULT_RESUME)


Saved best default hourly model: C:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\nn_model\artifacts\default_hourly\model.pt
Resume checkpoint: C:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\nn_model\artifacts\default_hourly\resume_latest.pt


## 15 · Evaluate using the group’s shared script

Export all validation impressions as Top-500 recommendations and run the same `cg_evaluation.py` used by the other candidate-generation baselines. The official test remains untouched.


In [15]:
OUTPUT_TOP500 = DEFAULT_DIR / 'validation_top500.parquet'
evaluator_writer = CandidateWriter(OUTPUT_TOP500)
try:
    FINAL_RANKS = click_ranks(ModelScorer(DEFAULT_MODEL), VALID,
                             VALID_CLICK_OFF, VALID_CLICK_FLAT,
                             k=max(CFG.TOP_MS), writer=evaluator_writer)
finally:
    evaluator_writer.close()
row_count = pq.ParquetFile(OUTPUT_TOP500).metadata.num_rows
check('evaluation', 'complete validation output rows',
      row_count, len(VALID) * max(CFG.TOP_MS))
path = DEFAULT_DIR / 'validation_cg_evaluation.csv'
GROUP_METRICS = evaluator.evaluate(
    OUTPUT_TOP500, model_name='neural_two_tower_default_hourly', m_values=CFG.TOP_MS)
GROUP_METRICS.write_csv(path)
ours = recall_table(FINAL_RANKS, VALID_CLICK_OFF, CFG.TOP_MS)
theirs = GROUP_METRICS.to_pandas().set_index('m')
gap = max(abs(float(row.recall_impression_mean) -
              float(theirs.loc[row.m, 'recall_impression_mean']))
          for row in ours.itertuples())
check('evaluation', 'notebook recall equals shared evaluator', gap < 1e-9, True)
display(GROUP_METRICS)
print('Saved:', OUTPUT_TOP500, 'and', path)


model,split,m,recall_pooled,recall_impression_mean,hit_rate,impressions,relevant_clicks,retrieved_relevant,mean_candidates,mean_eligible_pool,eligible_click_coverage
str,str,i64,f64,f64,f64,i64,i64,i64,f64,f64,f64
"""neural_two_tower_default_hourl…","""validation""",50,0.643884,0.669584,0.744698,334913,514658,331380,50.0,99816.190673,1.0
"""neural_two_tower_default_hourl…","""validation""",100,0.764123,0.779191,0.835142,334913,514658,393262,100.0,99816.190673,1.0
"""neural_two_tower_default_hourl…","""validation""",200,0.858207,0.864689,0.900365,334913,514658,441683,200.0,99816.190673,1.0
"""neural_two_tower_default_hourl…","""validation""",500,0.937391,0.938521,0.954851,334913,514658,482436,500.0,99816.190673,1.0


Saved: C:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\nn_model\artifacts\default_hourly\validation_top500.parquet and C:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\nn_model\artifacts\default_hourly\validation_cg_evaluation.csv


In [ ]:
# ===== Extra · Evaluate the trained default model on the official TEST split =====.

TEST_SPLITS = ("train", "validation", "test")


visits = (pl.concat([pl.read_parquet(STAGE1 / f"impressions_{s}.parquet", columns=["user_code", "timestamp"])
                     for s in TEST_SPLITS])
          .unique().sort(["user_code", "timestamp"]))
visit_gaps = visits.with_columns(
    gap_hours=(pl.col("timestamp") - pl.col("timestamp").shift(1).over("user_code")).dt.total_seconds() / 3600.0)


last_test = pl.scan_parquet(STAGE1 / "candidates_test.parquet").select(pl.col("timestamp").max()).collect().item()
N_HOURS = int((pd.Timestamp(last_test).value - T0_NS) // HOUR_NS) + 1
hourly = (pl.concat([pl.scan_parquet(STAGE1 / f"candidates_{s}.parquet")
                     .select("news_code", "clicked",
                             ((pl.col("timestamp").dt.epoch("ns") - T0_NS) // HOUR_NS).alias("h"))
                     for s in TEST_SPLITS])
          .group_by("news_code", "h")
          .agg(exp=pl.len(), clk=pl.col("clicked").cast(pl.Int64).sum())
          .collect())
h_idx = hourly["h"].to_numpy().astype(np.int64)
r_idx = to_rows(hourly["news_code"].to_numpy())
EXPOSURES = np.zeros((N_HOURS, N_ROWS), dtype=np.int32)
CLICKS = np.zeros((N_HOURS, N_ROWS), dtype=np.int32)
np.add.at(EXPOSURES, (h_idx, r_idx), hourly["exp"].to_numpy())
np.add.at(CLICKS, (h_idx, r_idx), hourly["clk"].to_numpy())
del hourly, h_idx, r_idx

CUM_EXP = np.zeros((N_HOURS + 1, N_ROWS), dtype=np.int32)
CUM_CLK = np.zeros((N_HOURS + 1, N_ROWS), dtype=np.int32)
np.cumsum(EXPOSURES, axis=0, out=CUM_EXP[1:])
np.cumsum(CLICKS, axis=0, out=CUM_CLK[1:])
TOTAL_EXP = CUM_EXP.sum(axis=1, dtype=np.int64)
TOTAL_CLK = CUM_CLK.sum(axis=1, dtype=np.int64)
LAST_SEEN = np.full((N_HOURS, N_ROWS), -1, dtype=np.int16)
running = np.full(N_ROWS, -1, dtype=np.int16)
for h in range(N_HOURS):
    LAST_SEEN[h] = running
    running[EXPOSURES[h] > 0] = h
AVAIL_HOURS = (np.minimum(avail_ns, T0_NS + 10 * N_HOURS * HOUR_NS) - T0_NS) / HOUR_NS

print("Hourly states now cover", N_HOURS, "hours (train + validation + test day)")


TEST = load_requests("test")
TEST.ctx = ((TEST.ctx - CONTEXT_MEAN) / CONTEXT_STD).astype(np.float32)
TEST_CLICK_OFF, TEST_CLICK_FLAT, _ = clicks_by_request(TEST, "test")
print(f"Test requests: {len(TEST):,} | test clicks: {len(TEST_CLICK_FLAT):,}")


if "DEFAULT_MODEL" not in globals():
    saved = torch.load(DEFAULT_DIR / "model.pt", map_location=DEVICE, weights_only=False)
    DEFAULT_MODEL = TwoTowerRetriever(CFG).to(DEVICE)
    DEFAULT_MODEL.load_state_dict(saved["model_state"])
DEFAULT_MODEL.eval()


test_evaluator = CandidateEvaluator(root=REPO_ROOT, split="test")
check("evaluation", "Stage 1 test impressions = our test requests", test_evaluator.impressions.height, len(TEST))
TEST_TOP500 = DEFAULT_DIR / "test_top500.parquet"
started = time.time()
writer = CandidateWriter(TEST_TOP500)
try:
    TEST_RANKS = click_ranks(ModelScorer(DEFAULT_MODEL), TEST, TEST_CLICK_OFF, TEST_CLICK_FLAT,
                             k=max(CFG.TOP_MS), writer=writer)
finally:
    writer.close()
TEST_METRICS = test_evaluator.evaluate(TEST_TOP500, model_name="neural_two_tower_default_hourly",
                                       m_values=CFG.TOP_MS)
TEST_METRICS.write_csv(DEFAULT_DIR / "test_cg_evaluation.csv")
print(f"Test evaluation took {(time.time() - started) / 60:.1f} min")


validation_metrics = pl.read_csv(DEFAULT_DIR / "validation_cg_evaluation.csv")
summary = (pl.concat([validation_metrics, TEST_METRICS], how="diagonal_relaxed")
           .select("split", "m", "recall_impression_mean", "recall_pooled", "hit_rate")
           .sort(["m", "split"]))
display(summary.to_pandas())
print("Saved:", TEST_TOP500, "and", DEFAULT_DIR / "test_cg_evaluation.csv")

Hourly states now cover 168 hours (train + validation + test day)
Test requests: 376,471 | test clicks: 574,845
Test evaluation took 4.4 min


,split,m,recall_impression_mean,recall_pooled,hit_rate
0,test,50,0.740227,0.710049,0.805980
1,validation,50,0.669584,0.643884,0.744698
2,test,100,0.824760,0.807096,0.872723
3,validation,100,0.779191,0.764123,0.835142
4,test,200,0.895701,0.887471,0.925051
5,validation,200,0.864689,0.858207,0.900365
6,test,500,0.953162,0.950501,0.966462
7,validation,500,0.938521,0.937391,0.954851


Saved: C:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\nn_model\artifacts\default_hourly\test_top500.parquet and C:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\nn_model\artifacts\default_hourly\test_cg_evaluation.csv
